In [1]:
from training import count_parameters, TrainConfig, train_waitk_student, TranslationDataset, save_and_log_checkpoint, load_training_checkpoint
import mlflow
import torch
import json
from evaluation import SimulMTEvaluator, MTQualityScorer, WaitKTransformerAdapter
from model_classes import WaitKTransformerMT, SimulTransformerConfig

In [2]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("simulmt_waitk_transformer_distillation")

from transformers import AutoTokenizer

teacher_name = "./models/nllb_teacher"
tokenizer = AutoTokenizer.from_pretrained(teacher_name)

In [3]:
model_cfg = SimulTransformerConfig(
    vocab_size=len(tokenizer),
    d_model=512,
    nhead=8,
    num_encoder_layers=6,
    num_decoder_layers=6,
    dim_feedforward=2048,
    dropout=0.1,
    max_seq_len=64,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

In [5]:
transformer = WaitKTransformerMT(model_cfg)
count_parameters(transformer)

Total parameters:     306,558,976
Trainable parameters: 306,558,976
Frozen parameters:    0


{'total': 306558976, 'trainable': 306558976, 'frozen': 0}

In [6]:
train_cfg = TrainConfig(
    epochs=8,
    short_epochs=False,
    batches_per_epoch=2000,
    batch_size=192,
    gradient_accumulation_steps=8,

    wait_k=[3, 5, 7, 9, 11],

    use_kl_loss=False,
    use_dataset_ce_loss=True,

    kl_weight=1.0,
    dataset_ce_weight=1.0,

    lr=2e-5,
    use_amp=True,
)

In [7]:
dataset = TranslationDataset("./data/train_dataset.hdf5")

In [ ]:
train_waitk_student(
    student=transformer,
    train_dataset=dataset,
    model_cfg=model_cfg,
    train_cfg=train_cfg,
    device="cuda",
    checkpoint_name_prefix="transformer",
    resume_from_checkpoint="./checkpoints/transformer_epoch_4.pt"
)

In [4]:
eval_dataset = TranslationDataset(
    "./data/val_dataset.hdf5",
    lazy=False,
)

In [5]:
transformer = load_training_checkpoint("./models/transformer_wait10.pt", SimulTransformerConfig, WaitKTransformerMT)[0]

In [6]:
adapter = WaitKTransformerAdapter(
    model=transformer,
    tokenizer=tokenizer,
    name="transformer_wait_k",
    device="cuda",
    use_amp=True,
)

evaluator = SimulMTEvaluator(
    tokenizer=tokenizer,
    quality_scorer=MTQualityScorer(),
)

for k in [3, 5, 7, 9, 11]:
    result = evaluator.evaluate(
        model=adapter,
        dataset=eval_dataset,
        wait_k=k,
        batch_size=1024,
        max_new_tokens=63
    )
    
    print(result.metrics)
    
    with open(f"./metrics/transformer_k{k}.json", "w") as file:
        json.dump(result.metrics, file)

Validating transformer_wait_k, wait_k=3:   0%|          | 0/303 [00:00<?, ?it/s]

{'BLEU': 26.51217614723366, 'chrF++': 50.60682689603566, 'TER': 66.36694656876138, 'AP': 0.6660709003412466, 'AL': 4.457119921405012, 'DAL': 5.2114228167787395, 'LAAL': 4.817443593593854, 'ATD_text': 3.158894270754816, 'total_time_sec': 979.7127206869999, 'ms_per_sentence': 3.166389970223974, 'target_tokens_per_sec': 9142.17077197827, 'source_tokens_per_sec': 7944.432929830875, 'first_token_latency_sec': 3.116273010637119, 'peak_gpu_memory_mb': 2891.00537109375, 'generation_total_time_sec': 942.2138807370022, 'generation_ms_per_sentence': 3.0451953095795297, 'generation_target_tokens_per_sec': 9506.016821779407}


Validating transformer_wait_k, wait_k=5:   0%|          | 0/303 [00:00<?, ?it/s]

{'BLEU': 28.769817528493657, 'chrF++': 51.84776821856022, 'TER': 63.293108992570055, 'AP': 0.7287185663213963, 'AL': 6.306387438869035, 'DAL': 7.074938431202773, 'LAAL': 6.227929738394046, 'ATD_text': 4.378099030327806, 'total_time_sec': 1003.6019329889998, 'ms_per_sentence': 3.243598891402992, 'target_tokens_per_sec': 8744.39726701502, 'source_tokens_per_sec': 7755.327828852747, 'first_token_latency_sec': 3.1882177971003594, 'peak_gpu_memory_mb': 2891.00537109375, 'generation_total_time_sec': 963.946222163002, 'generation_ms_per_sentence': 3.1154333155457223, 'generation_target_tokens_per_sec': 9104.132365712005}


Validating transformer_wait_k, wait_k=7:   0%|          | 0/303 [00:00<?, ?it/s]

{'BLEU': 29.15972820267647, 'chrF++': 52.1051297369162, 'TER': 62.37521065191479, 'AP': 0.7830317454912569, 'AL': 8.127549068749182, 'DAL': 8.898381063434202, 'LAAL': 7.507694712560046, 'ATD_text': 5.5479142130859325, 'total_time_sec': 1027.307233174, 'ms_per_sentence': 3.3202134164183446, 'target_tokens_per_sec': 8455.920214988568, 'source_tokens_per_sec': 7576.3722367188975, 'first_token_latency_sec': 3.2599117223853082, 'peak_gpu_memory_mb': 2891.00537109375, 'generation_total_time_sec': 985.617067557002, 'generation_ms_per_sentence': 3.185472568944126, 'generation_target_tokens_per_sec': 8813.59331726224}


Validating transformer_wait_k, wait_k=9:   0%|          | 0/303 [00:00<?, ?it/s]

{'BLEU': 29.41227359889846, 'chrF++': 52.22526183989654, 'TER': 61.863045428279214, 'AP': 0.8274705133932596, 'AL': 9.88989582080969, 'DAL': 10.645150118598098, 'LAAL': 8.62921553453847, 'ATD_text': 6.598839580350782, 'total_time_sec': 1048.4227306430002, 'ms_per_sentence': 3.388457808871724, 'target_tokens_per_sec': 8235.326026081753, 'source_tokens_per_sec': 7423.782194446038, 'first_token_latency_sec': 3.328208020951964, 'peak_gpu_memory_mb': 2891.00537109375, 'generation_total_time_sec': 1006.2614807110003, 'generation_ms_per_sentence': 3.252194436866941, 'generation_target_tokens_per_sec': 8580.377134081838}


Validating transformer_wait_k, wait_k=11:   0%|          | 0/303 [00:00<?, ?it/s]

{'BLEU': 29.473671930745805, 'chrF++': 52.222402625305875, 'TER': 61.59873016487063, 'AP': 0.8632399961447056, 'AL': 11.568563721590648, 'DAL': 12.2930216211215, 'LAAL': 9.600158777985051, 'ATD_text': 7.5149718726080454, 'total_time_sec': 1068.6120153910006, 'ms_per_sentence': 3.453708721085293, 'target_tokens_per_sec': 8041.17198406748, 'source_tokens_per_sec': 7283.524691748986, 'first_token_latency_sec': 3.394643980237157, 'peak_gpu_memory_mb': 2891.00537109375, 'generation_total_time_sec': 1026.3444075059924, 'generation_ms_per_sentence': 3.317101604686314, 'generation_target_tokens_per_sec': 8372.32895425489}
